# Data Exploration & Visualization

1. Dataset statistics (file counts, formats, sizes)
2. Class distribution (COVID vs Non-COVID)
3. Raw image samples gallery
4. LR vs HR paired comparison
5. Pixel intensity / RGB channel analysis
6. Original resolution analysis
7. Bicubic upscale baseline vs ground truth

In [ ]:
!pip install torch torchvision pillow tqdm matplotlib numpy -q

In [ ]:
import os
import sys
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter
from PIL import Image
from torchvision import transforms

# Add project root to path so we can import model.py
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))
from model import CTScanDataset

plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (14, 6)

#  DATASET PATH 
DATASET_PATH = r"C:\Users\NarisettyDharmaTeja\Downloads\Dataset"

## 1. Dataset Statistics
Scan the dataset directory to gather file counts, format distribution, and file size information.

In [ ]:
# Collect all image paths, extensions, sizes, and parent folders
all_paths, extensions, file_sizes, class_labels = [], [], [], []

for root, _, files in os.walk(DATASET_PATH):
    for f in files:
        fpath = os.path.join(root, f)
        ext = os.path.splitext(f)[1].lower()
        if ext in (".png", ".jpg", ".jpeg", ".bmp", ".tiff"):
            all_paths.append(fpath)
            extensions.append(ext)
            file_sizes.append(os.path.getsize(fpath) / 1024)  # KB
            # Use immediate parent folder name as class label
            class_labels.append(os.path.basename(root))

print(f"{'Total images found:':<30} {len(all_paths)}")
print(f"{'Unique classes/folders:':<30} {len(set(class_labels))}")
print(f"{'Format distribution:':<30} {dict(Counter(extensions))}")
print(f"{'Avg file size:':<30} {np.mean(file_sizes):.1f} KB")
print(f"{'Min / Max file size:':<30} {np.min(file_sizes):.1f} KB / {np.max(file_sizes):.1f} KB")

## 2. Class Distribution
Bar chart showing the number of images per class (subdirectory).

In [ ]:
class_counts = Counter(class_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart – class distribution
classes = list(class_counts.keys())
counts  = list(class_counts.values())
colors  = plt.cm.Set2(np.linspace(0, 1, len(classes)))

axes[0].bar(classes, counts, color=colors, edgecolor="black")
axes[0].set_title("Class Distribution", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Number of Images")
for i, (c, v) in enumerate(zip(classes, counts)):
    axes[0].text(i, v + 5, str(v), ha="center", fontweight="bold")

# Histogram – file size distribution
axes[1].hist(file_sizes, bins=50, color="steelblue", edgecolor="black", alpha=0.8)
axes[1].set_title("File Size Distribution", fontsize=14, fontweight="bold")
axes[1].set_xlabel("File Size (KB)")
axes[1].set_ylabel("Frequency")
axes[1].axvline(np.mean(file_sizes), color="red", linestyle="--", label=f"Mean: {np.mean(file_sizes):.1f} KB")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Original Image Resolution Analysis
Inspect the native resolutions of images in the dataset before any resizing transforms.

In [ ]:
# Sample up to 500 images for resolution analysis (fast)
sample_paths = random.sample(all_paths, min(500, len(all_paths)))
widths, heights = [], []

for p in sample_paths:
    with Image.open(p) as img:
        w, h = img.size
        widths.append(w)
        heights.append(h)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: width vs height
axes[0].scatter(widths, heights, alpha=0.4, s=15, c="teal", edgecolors="black", linewidths=0.3)
axes[0].set_title("Original Image Resolutions", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Width (px)")
axes[0].set_ylabel("Height (px)")
axes[0].axhline(256, color="red", linestyle="--", alpha=0.6, label="HR target (256)")
axes[0].axvline(256, color="red", linestyle="--", alpha=0.6)
axes[0].legend()

# Histogram of total pixel counts
total_pixels = [w * h for w, h in zip(widths, heights)]
axes[1].hist(total_pixels, bins=40, color="coral", edgecolor="black", alpha=0.8)
axes[1].set_title("Total Pixel Count Distribution", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Total Pixels (W × H)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

print(f"Resolution range:  {min(widths)}×{min(heights)}  →  {max(widths)}×{max(heights)}")
print(f"Median resolution: {int(np.median(widths))}×{int(np.median(heights))}")

## 4. Raw Image Gallery
Display a 4×4 grid of randomly sampled CT scan images from the dataset.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
samples = random.sample(all_paths, 16)

for ax, path in zip(axes.flatten(), samples):
    img = Image.open(path).convert("RGB")
    ax.imshow(img)
    # Show class label and original size
    label = os.path.basename(os.path.dirname(path))
    w, h = img.size
    ax.set_title(f"{label}\n{w}*{h}", fontsize=9)
    ax.axis("off")

fig.suptitle("Random CT Scan Samples from Dataset", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 5. LR ↔ HR Paired Comparison
Show how the dataset creates paired Low-Resolution (64×64) and High-Resolution (256×256) images side by side.

In [ ]:
from torch.utils.data import DataLoader

dataset = CTScanDataset(DATASET_PATH)
loader  = DataLoader(dataset, batch_size=8, shuffle=True)

lr_batch, hr_batch = next(iter(loader))

fig, axes = plt.subplots(2, 8, figsize=(20, 6))

for i in range(8):
    # Top row: LR (64*64)
    axes[0, i].imshow(lr_batch[i].permute(1, 2, 0).numpy())
    axes[0, i].set_title(f"LR 64*64", fontsize=9)
    axes[0, i].axis("off")

    # Bottom row: HR (256*256)
    axes[1, i].imshow(hr_batch[i].permute(1, 2, 0).numpy())
    axes[1, i].set_title(f"HR 256*256", fontsize=9)
    axes[1, i].axis("off")

fig.suptitle("Low-Resolution (top)  vs  High-Resolution (bottom)", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"LR tensor shape: {lr_batch.shape}  |  HR tensor shape: {hr_batch.shape}")
print(f"LR pixel range:  [{lr_batch.min():.3f}, {lr_batch.max():.3f}]")
print(f"HR pixel range:  [{hr_batch.min():.3f}, {hr_batch.max():.3f}]")

## 6. Pixel Intensity & RGB Channel Analysis
Examine the pixel value distributions for both LR and HR images across R, G, B channels.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
channel_names = ["Red", "Green", "Blue"]
channel_colors = ["red", "green", "blue"]

# Use a single sample for detailed channel analysis
lr_img = lr_batch[0].numpy()  # (3, 64, 64)
hr_img = hr_batch[0].numpy()  # (3, 256, 256)

for c in range(3):
    # LR channel histograms
    axes[0, c].hist(lr_img[c].ravel(), bins=50, color=channel_colors[c], alpha=0.7, edgecolor="black")
    axes[0, c].set_title(f"LR – {channel_names[c]} Channel", fontsize=12, fontweight="bold")
    axes[0, c].set_xlabel("Pixel Value")
    axes[0, c].set_ylabel("Frequency")
    axes[0, c].set_xlim(0, 1)

    # HR channel histograms
    axes[1, c].hist(hr_img[c].ravel(), bins=50, color=channel_colors[c], alpha=0.7, edgecolor="black")
    axes[1, c].set_title(f"HR – {channel_names[c]} Channel", fontsize=12, fontweight="bold")
    axes[1, c].set_xlabel("Pixel Value")
    axes[1, c].set_ylabel("Frequency")
    axes[1, c].set_xlim(0, 1)

fig.suptitle("RGB Channel Pixel Distributions:  LR (top)  vs  HR (bottom)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Per-channel statistics across the batch
for name, batch in [("LR", lr_batch), ("HR", hr_batch)]:
    print(f"\n{name} batch statistics (8 images):")
    for c, ch in enumerate(channel_names):
        vals = batch[:, c].numpy()
        print(f"  {ch:6s} →  mean={vals.mean():.4f}  std={vals.std():.4f}  min={vals.min():.4f}  max={vals.max():.4f}")

## 7. Bicubic Baseline vs Ground Truth
Compare naive bicubic upscaling (64→256) with the ground truth HR to establish a quality baseline.
This is what the Generator must **beat**.

In [ ]:
bicubic_upsample = transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.BICUBIC)

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
col_titles = ["LR (64*64)", "Bicubic 4* (256*256)", "Ground Truth HR (256*256)"]

for i in range(4):
    lr_tensor = lr_batch[i]
    hr_tensor = hr_batch[i]

    # Bicubic upscale: LR → 256*256
    bicubic_tensor = bicubic_upsample(lr_tensor)

    axes[i, 0].imshow(lr_tensor.permute(1, 2, 0).numpy())
    axes[i, 1].imshow(bicubic_tensor.permute(1, 2, 0).clamp(0, 1).numpy())
    axes[i, 2].imshow(hr_tensor.permute(1, 2, 0).numpy())

    for j in range(3):
        axes[i, j].axis("off")
        if i == 0:
            axes[i, j].set_title(col_titles[j], fontsize=12, fontweight="bold")

    # Compute PSNR between bicubic and GT
    mse_val = ((bicubic_tensor - hr_tensor) ** 2).mean().item()
    psnr = 10 * np.log10(1.0 / mse_val) if mse_val > 0 else float("inf")
    axes[i, 1].set_xlabel(f"PSNR: {psnr:.2f} dB", fontsize=10)

fig.suptitle("Bicubic Upscale Baseline  vs  Ground Truth", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Mean Image & Variance Map
Compute the per-pixel mean and variance across the dataset to understand the spatial structure of CT scans.

In [ ]:
# Compute running mean and variance over a subset of the dataset
N_SAMPLES = min(200, len(dataset))
resize_tf = transforms.Compose([transforms.Resize((256, 256)), transforms.ToTensor()])

running_sum = np.zeros((3, 256, 256), dtype=np.float64)
running_sq  = np.zeros((3, 256, 256), dtype=np.float64)

for i in range(N_SAMPLES):
    img = Image.open(all_paths[i]).convert("RGB")
    t = resize_tf(img).numpy().astype(np.float64)
    running_sum += t
    running_sq  += t ** 2

mean_img = running_sum / N_SAMPLES
var_img  = (running_sq / N_SAMPLES) - mean_img ** 2

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(np.clip(mean_img.transpose(1, 2, 0), 0, 1))
axes[0].set_title(f"Mean Image (over {N_SAMPLES} samples)", fontsize=13, fontweight="bold")
axes[0].axis("off")

# Show variance as grayscale (averaged across channels)
var_gray = var_img.mean(axis=0)
im = axes[1].imshow(var_gray, cmap="hot")
axes[1].set_title(f"Pixel Variance Map", fontsize=13, fontweight="bold")
axes[1].axis("off")
plt.colorbar(im, ax=axes[1], fraction=0.046)

plt.tight_layout()
plt.show()